In [ ]:
import ipdb # <- трасировка и точки останова

In [ ]:
from header import __root__
from src import gs

In [ ]:
import importlib
import asyncio
from pathlib import Path
from types import SimpleNamespace
from typing import Optional, Dict, Any, List


from src.llm.gemini import GoogleGenerativeAi # Unused, but kept
from src.endpoints.prestashop.product_fields import ProductFields
from src.endpoints.prestashop.product_async import PrestaProductAsync
from src.endpoints.prestashop.product import PrestaProduct
from src.suppliers.graber_via_pydoll import Config as GraberConfig, Graber as SupplierGraber

from src.utils.file import read_text_file, save_text_file, get_filenames_from_directory
from src.utils.jjson import j_loads, j_loads_ns, j_dumps # j_dumps unused
from src.utils.image import get_image_bytes, get_raw_image_data 
from src.utils.printer import pprint as print
from src.logger.logger import logger

In [ ]:
browser: Chrome = None
page:'Page' = None

In [ ]:
# --- config.py ---
class Config:
    """Script-wide configuration (not supplier-specific).

    ----------------------------------------------------------
    По умолчанию используется SANDBOX/davidka/scenarios/*.json
    и престашоп store.davidka.net
    
    """

    ENDPOINT: Path = __root__ / 'SANDBOX' / 'davidka'
    SCENARIOS_DIR: Path = ENDPOINT / 'scenarios'
    
    SUPPLIERS_ENDPOINT: Path = __root__ / 'src' / 'suppliers' / 'suppliers_list'

    PRESTA_API_KEY: str = gs.credentials.prestashop.store_davidka_net.api_key
    PRESTA_API_DOMAIN: str = gs.credentials.prestashop.store_davidka_net.api_domain
    
    @property
    def scenarios_files(self) -> List[str]:
        return get_filenames_from_directory(self.SCENARIOS_DIR)
        
# --- end config.py ---

In [ ]:
'''


async def fetch_product_fields(page: Page, required_fields:Optional[list] = None) -> ProductFields:
    """Grab product fields."""

    ENDPOINT: Path = __root__ / 'src' / 'suppliers' / 'suppliers_list' / 'aliexpress' 

    required_fields:list = ['id_supplier',                                                              
                         'name',
                         'price',
                         'reference',
                         'description',
                         'description_short',
                         'default_image_url',
                         'local_image_path',]

    locator:SimpleNamespace = j_loads_ns(ENDPOINT / 'locators' / 'product.json')

    async def save_local_image(f) -> bool:
        """Fetch and save an image locally.

        Функция получает `URL` картинки или байты изображения, сохраняет изображение в формате `PNG` в директории `tmp` 
        и устанавливает путь к сохранённой картинке в поле `local_image_path`.
        """
        try:
            # Получаем результат из локатора как `bytes` или `str`(url)
            image_url:str = f.default_image_url
            img_path:Path = Path(gs.path.tmp / f'{f.id_supplier}_{f.reference}.png')
            await save_image_from_url_async(image_url, img_path)
            return img_path
        except Exception as ex:
            logger.error(f'Ошибка сохранения изображения в поле `local_image_path`', ex)
            ...
            return None


    f:ProductFields = ProductFields()

    f.id_supplier = locator.id_supplier.attribute
    f.name = await execute_locator(page, locator.name)
    f.reference = page.current_url.split("/item/")[1].split(".html")[0]
    f.price = await execute_locator(page, locator.price)

    if 'description' in required_fields:
        f.description = await execute_locator(page, locator.description)
    if 'description_short' in required_fields:
        f.description_short = await execute_locator(page, locator.description_short)
    if 'default_image_url' in required_fields:
        f.default_image_url = await execute_locator(page, locator.default_image_url)
    if 'local_image_path' in required_fields:
        f.local_image_path = await save_local_image(f)

    return f

async def grab_product_page( page: Page, product_url: str, required_fields:Optional[list] = None) -> ProductFields:
    """
    Загружает страницу товара по URL и возвращает структуру данных ProductFields.
    Поддерживаются входные URL формата:
        //he.aliexpress_com.com/item/
        https://he.aliexpress_com.com/item/
        he.aliexpress_com.com/item/
    """

    if product_url.startswith('//'):
        url = f'https:{product_url}'
    elif product_url.startswith('http://') or product_url.startswith('https://'):
        url = product_url
    else:
        url = f'https://{product_url.lstrip("/")}'

    await page.go_to(url)
    return await fetch_product_fields(product_url, page, required_fields or Config.required_fields)

async def get_product_urls_from_category_page(category_url:str, locator:SimpleNamespace, page: Page) -> List[str]:
    """Get product URLs from the current page.
   Отдельная функция для каждого поставщика, так как локаторы могут отличаться.
    """
    await page.go_to(category_url)
    product_urls = await execute_locator(page, locator)
    return product_urls


In [ ]:
if not browser:
    browser = Chrome()  
    await browser.start()
    
if not page:
    page = await browser.get_page()

In [ ]:
# ПРАВИЛЬНО:
async def __save_to_prestashop_async(f:ProductFields):
    """"""
    async with Config.presta_product_async as presta_product_async:
        # Теперь presta_product_api.client инициализирован
        result = await presta_product_async.add_new_product_async(f)

# Плохо
async def save_to_prestashop_async(f:ProductFields):
    """"""
    p = PrestaProduct(PRESTA_API_KEY,PRESTA_API_DOMAIN)
    print(f.to_dict())
    ipdb.set_trace()
    result = await p.add_new_product_async(f)
    

#### process supplier

In [ ]:
def get_graber(supplier_alias) -> Any:
    graber_module_path:str  = f"src.suppliers.suppliers_list.{supplier_alias}.graber_via_pydoll"
    try:
        graber: 'Graber' = importlib.import_module(graber_module_path)
        return graber
    except Exception as ex:
        logger.error(f"Failed to import module `graber` '{supplier_prefix}'", ex)
        return None    

In [ ]:
async def process_supplier(supplier_prefix:str, page: 'Page', product_url:Optional[str] = None ) -> bool:
    """Название файла JSON соответствуют `supplier_prefix`, а  названия папок в системе - `supplier_alias` """
    ...

    graber = get_graber(supplier_prefix)
    
    if not graber:
        return False

    if product_url: # <- обработка одной ссылки
        f:ProductFields = await graber.grab_product_page(page, product_url, required_fields)
        return await save_to_prestashop_async(f)

    for f in graber.yield_all_scenarios()
        await save_to_prestashop_async(f)
    

In [ ]:
await process_supplier('aliexpress', page)

---
---

### Точечные проверки

In [ ]:
ENDPOINT: Path = __root__ / 'src' / 'suppliers' / 'suppliers_list' / 'aliexpress'
product_locator = j_loads_ns(ENDPOINT / 'locators' / 'product.json')
category_locator = j_loads_ns(ENDPOINT / 'locators' / 'category.json')


In [ ]:

#product_url:str = 'https://he.aliexpress_com.com/item/1005004869497167.html?algo_pvid=5bcf218a-5626-40fd-99ed-9ede265fa590&algo_exp_id=5bcf218a-5626-40fd-99ed-9ede265fa590-0&pdp_ext_f=%7B%22order%22%3A%22172%22%2C%22eval%22%3A%221%22%7D&pdp_npi=4%40dis%21ILS%21812.41%21568.70%21%21%21224.18%21156.93%21%40212e508f17497439898337727ea5e6%2112000030824283710%21sea%21IL%210%21ABX&curPageLogUid=G8tFBGu1NjyI&utparam-url=scene%3Asearch%7Cquery_from%3A#nav-specification'
#category_url = 'https://www.aliexpress_com.com/w/wholesale-industrial-servo-motors.html?spm=a2g0o.productlist.search.0'
#res = await get_product_urls_from_category_page(category_url, category_locator.product_links, page)


In [ ]:
#res = await execute_locator(page, category_locator.product_links)

In [ ]:
graber_module_path:str  = f"src.suppliers.suppliers_list.{supplier_alias}.graber_via_pydoll"
graber_module = importlib.import_module(graber_module_path)
graber = graber_module.Graber(supplier_prefix)

if not browser:
    browser = Chrome()  
    await browser.start()
    
if not page:
    page = await browser.get_page()
    
await graber.grab_product_page(page, product_url)

In [ ]:
required_fields:list = ['id_supplier',                                                              
                     'name',
                     'price',
                     'reference',
                     'description',
                     'description_short',
                     'default_image_url']

f:ProductFields = ProductFields()

f.id_supplier = locator.id_supplier.attribute
f.name = await execute_locator(page, locator.name)
f.reference = product_url.split("/item/")[1].split(".html")[0]

f.price = await execute_locator(page, locator.price)


if 'description' in required_fields:
    f.description = await execute_locator(page, locator.description)
if 'description_short' in required_fields:
    f.description_short = await execute_locator(page, locator.description_short)
if 'default_image_url' in required_fields:
    f.default_image_url = await execute_locator(page, locator.default_image_url)

In [ ]:
print(f.to_dict())

In [ ]:
await save_to_prestashop(f)